# Optigami — Origami RL Training (GRPO)

**Train an LLM to generate valid origami fold sequences using verifiable geometric rewards.**

Architecture:
- **Environment**: `env/` — CreaseGraph + Kawasaki/Maekawa/BLB verifiers + target matching
- **Policy model**: SpatialThinker (Qwen2.5-VL-7B) or vanilla Qwen2.5-VL-7B
- **Training**: Unsloth GRPO — model generates complete fold sequences, gets terminal reward
- **Tracking**: Trackio — real-time reward curves on HF Spaces

| Reward Component | Weight | What it measures |
|---|---|---|
| `progress` | 0.45 | Geometric crease coverage vs target |
| `economy` | 0.10 | Penalty for excess creases |
| `kawasaki` | 0.08 | Kawasaki theorem satisfaction |
| `maekawa` | 0.07 | Maekawa theorem satisfaction |
| `blb` | 0.05 | Big-Little-Big lemma |
| `anchored` | 0.05 | Valid anchor point usage |
| `completion` | +10.0 | Bonus when target reached |

## 1. Setup

**GPU**: H100 80GB (Northflank/CoreWeave) or A100/T4 (Colab)

Install dependencies. Unsloth handles efficient model loading + LoRA.

In [ ]:
%%capture
!pip install unsloth trackio shapely numpy datasets
!pip install --upgrade trl transformers

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuration

Choose base model and hyperparameters. Two options:
- **SpatialThinker** (`OX-PIXL/SpatialThinker-Qwen2.5-VL-7B`): Pre-trained for spatial reasoning via RL
- **Vanilla Qwen2.5-VL** (`unsloth/Qwen2.5-VL-7B-Instruct`): Standard vision-language model

We'll compare both to see which learns origami folding faster.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
# Toggle MODEL_NAME to switch between SpatialThinker and vanilla Qwen2.5-VL

MODEL_NAME = "OX-PIXL/SpatialThinker-Qwen2.5-VL-7B"
# MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct"  # uncomment for vanilla

MAX_SEQ_LENGTH = 2048
LORA_R = 32
LORA_ALPHA = 32
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 5e-6
N_GENERATIONS = 8       # completions sampled per prompt (GRPO group size)
MAX_FOLDS = 8           # max folds per episode
LEVEL = 1               # target difficulty (1=simple, 2=medium, 3=hard)
MAX_COMPLETION_LEN = 512
OUTPUT_DIR = "origami-grpo"

# Trackio — set your HF Space ID for live dashboard
TRACKIO_SPACE_ID = None  # e.g. "your-username/optigami-training"

print(f"Model: {MODEL_NAME}")
print(f"Config: {EPOCHS} epochs, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}")
print(f"GRPO: {N_GENERATIONS} generations, max_folds={MAX_FOLDS}, level={LEVEL}")

## 3. Clone Repo & Test Environment

Clone the optigami repo (skip if running locally) and verify the environment works.

In [ ]:
import os

# Clone repo if not already present (Colab/Northflank)
if not os.path.exists("env/environment.py"):
    !git clone https://huggingface.co/spaces/openenv-community/optigami /content/optigami 2>/dev/null || true
    os.chdir("/content/optigami")

# Verify env/ is accessible
from env.environment import OrigamiEnvironment
from env.rewards import compute_reward
from env.prompts import parse_fold_list

env = OrigamiEnvironment(mode="code_as_policy", max_steps=MAX_FOLDS)
print(f"Available targets: {env.available_targets()}")
print(f"Environment mode: {env.mode}")

In [ ]:
# ── Dry run: test reward function ───────────────────────────────────────────
# Verify rewards work before loading the model

import copy

def make_reward_fn(env_template, max_folds):
    """Reward function: clone env, run completion, return total reward."""
    def reward_fn(completions, prompts=None, **kwargs):
        rewards = []
        target_names = kwargs.get("target_names", [None] * len(completions))
        for completion, target_name in zip(completions, target_names):
            try:
                e = env_template.clone()
                e.reset(target_name=target_name)
                _, reward_dict, _, _ = e.step(completion)
                rewards.append(float(reward_dict["total"]))
            except Exception:
                rewards.append(-0.1)
        return rewards
    return reward_fn

reward_fn = make_reward_fn(env, MAX_FOLDS)

test_completions = [
    '<folds>[{"instruction": "Valley fold along horizontal center", "from": [0, 0.5], "to": [1, 0.5], "assignment": "V"}]</folds>',
    '<folds>[{"instruction": "Bad fold", "from": [0.3, 0.3], "to": [0.7, 0.7], "assignment": "V"}]</folds>',
    'not valid JSON',
]
target_names = ["half_horizontal"] * 3
rewards = reward_fn(test_completions, target_names=target_names)

for comp, r in zip(["perfect fold", "partial fold", "garbage"], rewards):
    print(f"  {comp}: reward = {r:.3f}")
print("\nReward function OK ✓")

## 4. Load Model + LoRA

Load the VL model with Unsloth's `FastVisionModel` (4-bit quantized) and apply LoRA adapters.

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Build Dataset

Generate prompts from all level-appropriate targets. Each prompt embeds the target crease pattern description and asks the model to output `<folds>[...]</folds>`.

In [ ]:
import random
from datasets import Dataset

def build_dataset(env, level=1):
    """Build training dataset of prompts from env targets."""
    all_names = env.available_targets()
    level_names = [
        n for n in all_names
        if env._targets[n].get("level", 1) == level
    ]
    if not level_names:
        level_names = all_names

    items = []
    for name in level_names:
        obs = env.reset(target_name=name)
        items.append({"prompt": obs["prompt"], "target_name": name})

    # Repeat each target 10x; ensure at least 50 examples
    repeat = max(10, (50 + len(items) - 1) // len(items))
    items = items * repeat
    random.shuffle(items)
    return items

dataset_items = build_dataset(env, level=LEVEL)
hf_dataset = Dataset.from_list(dataset_items)

print(f"Dataset: {len(dataset_items)} examples")
print(f"Targets in dataset: {sorted(set(d['target_name'] for d in dataset_items))}")
print(f"\nSample prompt (first 300 chars):\n{dataset_items[0]['prompt'][:300]}...")

## 6. Trackio Setup

Initialize Trackio for real-time training visualization. Trackio is a free W&B alternative that deploys a Gradio dashboard to HF Spaces.

In [ ]:
import trackio

# Initialize Trackio run
trackio_kwargs = {
    "project_name": "optigami",
    "run_name": f"grpo-{MODEL_NAME.split('/')[-1]}-level{LEVEL}",
}
if TRACKIO_SPACE_ID:
    trackio_kwargs["space_id"] = TRACKIO_SPACE_ID

trackio.init(**trackio_kwargs)
print(f"Trackio initialized: {trackio_kwargs['run_name']}")
if TRACKIO_SPACE_ID:
    print(f"Dashboard: https://huggingface.co/spaces/{TRACKIO_SPACE_ID}")

## 7. GRPO Training

Run GRPO with Trackio logging. The trainer samples `N_GENERATIONS` completions per prompt, computes rewards via the environment, and updates the policy using group-relative advantages.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# ── Per-component reward functions for detailed logging ─────────────────────
REWARD_COMPONENTS = ["kawasaki", "maekawa", "blb", "progress", "economy", "completion"]

def make_component_fn(env_template, component):
    """Create a reward function that returns a single component's value."""
    def component_fn(completions, target_name=None, **kwargs):
        target_names = target_name if isinstance(target_name, list) else [target_name] * len(completions)
        rewards = []
        for completion, tn in zip(completions, target_names):
            try:
                e = env_template.clone()
                e.reset(target_name=tn)
                _, reward_dict, _, _ = e.step(completion)
                rewards.append(float(reward_dict.get(component, 0.0)))
            except Exception:
                rewards.append(0.0)
        return rewards
    component_fn.__name__ = f"reward_{component}"
    return component_fn

# Main reward function (returns total reward)
def wrapped_reward_fn(completions, target_name=None, **kwargs):
    """Main reward function — extracts target_name from batch columns."""
    target_names = target_name if isinstance(target_name, list) else [target_name] * len(completions)
    return reward_fn(completions, target_names=target_names)

# Build list of all reward functions: [total, kawasaki, maekawa, blb, progress, economy, completion]
all_reward_fns = [wrapped_reward_fn] + [
    make_component_fn(env, c) for c in REWARD_COMPONENTS
]

# ── GRPO Config ─────────────────────────────────────────────────────────────
config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_completion_length=MAX_COMPLETION_LEN,
    num_generations=N_GENERATIONS,
    temperature=1.0,
    logging_steps=1,
    save_steps=50,
    report_to="trackio",
    run_name=f"grpo-{MODEL_NAME.split('/')[-1]}-level{LEVEL}",
)

trainer = GRPOTrainer(
    model=model,
    config=config,
    train_dataset=hf_dataset,
    reward_funcs=all_reward_fns,
    tokenizer=tokenizer,
)

print(f"Trainer ready. Starting GRPO training...")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {len(hf_dataset)} examples")
print(f"  Reward functions: total + {REWARD_COMPONENTS}")
print(f"  Logging to: Trackio")

In [ ]:
# ── Train! ──────────────────────────────────────────────────────────────────
trainer.train()

# Save model + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nTraining complete. Model saved to {OUTPUT_DIR}/")

## 8. Evaluation

Run the trained model on all targets and measure solve rates + average rewards.

In [ ]:
# ── Evaluate trained model on all targets ───────────────────────────────────
from transformers import TextStreamer

FastVisionModel.for_inference(model)

eval_results = {}
n_samples = 5  # generations per target

for target_name in env.available_targets():
    obs = env.reset(target_name=target_name)
    prompt = obs["prompt"]

    # Tokenize prompt
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

    target_rewards = []
    target_solved = 0

    for _ in range(n_samples):
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_COMPLETION_LEN,
            temperature=0.7,
            do_sample=True,
        )
        completion = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

        # Score completion
        e = env.clone()
        e.reset(target_name=target_name)
        try:
            _, reward_dict, _, _ = e.step(completion)
            total = float(reward_dict["total"])
            solved = reward_dict.get("completion", 0) > 0
        except Exception:
            total = -0.1
            solved = False

        target_rewards.append(total)
        if solved:
            target_solved += 1

    eval_results[target_name] = {
        "avg_reward": sum(target_rewards) / len(target_rewards),
        "max_reward": max(target_rewards),
        "solve_rate": target_solved / n_samples,
    }
    print(f"  {target_name:20s}  avg={eval_results[target_name]['avg_reward']:.3f}  "
          f"max={eval_results[target_name]['max_reward']:.3f}  "
          f"solved={target_solved}/{n_samples}")

# Summary
avg_solve = sum(r["solve_rate"] for r in eval_results.values()) / len(eval_results)
avg_reward = sum(r["avg_reward"] for r in eval_results.values()) / len(eval_results)
print(f"\nOverall: avg_reward={avg_reward:.3f}, solve_rate={avg_solve:.1%}")

# Log to Trackio
trackio.log({"eval/avg_reward": avg_reward, "eval/solve_rate": avg_solve})
for name, res in eval_results.items():
    trackio.log({f"eval/{name}_reward": res["avg_reward"], f"eval/{name}_solved": res["solve_rate"]})

## 9. A/B Comparison: SpatialThinker vs Vanilla Qwen2.5-VL

To compare both models, run this notebook twice:
1. First run with `MODEL_NAME = "OX-PIXL/SpatialThinker-Qwen2.5-VL-7B"`
2. Second run with `MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct"`

Both runs log to the same Trackio project (`optigami`) with different run names, so you can overlay the reward curves directly in the dashboard.

The cell below loads saved eval results from both runs for comparison (run after both trainings complete).

In [ ]:
# ── Save eval results for comparison ────────────────────────────────────────
import json

model_tag = MODEL_NAME.split("/")[-1]
eval_path = f"eval_results_{model_tag}_level{LEVEL}.json"

with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"Eval results saved to {eval_path}")

# ── Compare (run after both models are trained) ────────────────────────────
spatial_path = f"eval_results_SpatialThinker-Qwen2.5-VL-7B_level{LEVEL}.json"
vanilla_path = f"eval_results_Qwen2.5-VL-7B-Instruct_level{LEVEL}.json"

if os.path.exists(spatial_path) and os.path.exists(vanilla_path):
    with open(spatial_path) as f:
        spatial = json.load(f)
    with open(vanilla_path) as f:
        vanilla = json.load(f)

    print(f"\n{'Target':<22} {'SpatialThinker':>16} {'Vanilla Qwen':>16} {'Delta':>10}")
    print("-" * 66)
    for target in sorted(set(list(spatial.keys()) + list(vanilla.keys()))):
        s_r = spatial.get(target, {}).get("avg_reward", 0)
        v_r = vanilla.get(target, {}).get("avg_reward", 0)
        delta = s_r - v_r
        print(f"  {target:<20} {s_r:>14.3f}   {v_r:>14.3f}   {delta:>+8.3f}")

    s_avg = sum(r["avg_reward"] for r in spatial.values()) / len(spatial)
    v_avg = sum(r["avg_reward"] for r in vanilla.values()) / len(vanilla)
    print(f"\n  {'OVERALL':<20} {s_avg:>14.3f}   {v_avg:>14.3f}   {s_avg - v_avg:>+8.3f}")

    s_solve = sum(r["solve_rate"] for r in spatial.values()) / len(spatial)
    v_solve = sum(r["solve_rate"] for r in vanilla.values()) / len(vanilla)
    print(f"  {'Solve Rate':<20} {s_solve:>13.1%}   {v_solve:>13.1%}   {s_solve - v_solve:>+7.1%}")
else:
    print(f"Run both models to compare. Looking for:\n  {spatial_path}\n  {vanilla_path}")

## 10. Push to HuggingFace Hub (optional)

Upload the trained LoRA adapter to HF for deployment or further fine-tuning.

In [ ]:
# ── Push to HF Hub (uncomment and set your repo) ───────────────────────────
# from huggingface_hub import login
# login(token="hf_...")  # or use HF_TOKEN env var
#
# HF_REPO = "your-username/optigami-grpo-spatialthinker"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f"Model pushed to https://huggingface.co/{HF_REPO}")

trackio.finish()
print("Done! Check your Trackio dashboard for training curves.")